# Mixed-Precision Training (AMP)

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

Mixed-precision keeps the model parameters and master copy in FP32 but runs the forward/backward in FP16/BF16. With `autocast` + `GradScaler` you get roughly 2× speed-up and half the memory on modern GPUs.


## Mathematical Formulation

For each step:

1. $z = \text{autocast}(\text{forward}(x))$ in FP16/BF16.
2. Loss scaled: $\tilde L = s\,L$ to keep FP16 gradients out of underflow.
3. Unscale + step on FP32 master copy.


## Implementation


In [ ]:
import torch
import torch.nn as nn


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = nn.Sequential(nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
scaler = torch.cuda.amp.GradScaler(enabled=device == 'cuda')

def step(x, y):
    with torch.cuda.amp.autocast(enabled=device == 'cuda'):
        logits = model(x)
        loss = nn.functional.cross_entropy(logits, y)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    opt.zero_grad()
    return loss.item()


## Experiment


In [ ]:
x = torch.randn(64, 256, device=device)
y = torch.randint(0, 10, (64,), device=device)
print('loss step 0:', step(x, y))
print('loss step 1:', step(x, y))


## Discussion

- BF16 (Ampere+ GPUs) often needs no loss scaling because its exponent range matches FP32.
- Disable autocast for ops that are numerically sensitive (layer-norm reductions, softmax in logits with extreme values).
- Always check that loss is finite — NaNs are the canonical AMP failure mode.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
